# 📑Tutorial-6: Distinguisher Attack on AES Software Implementation

**🔍distinguisher**

When performing side-channel attacks on cryptographic algorithms to recover keys, it is generally necessary to guess the key and use a distinguisher to identify the correct key. Commonly used distinguishers include:

- **✅CPA distinguisher**
- **✅DPA distinguisher**
- **✅ANOVA-SNR-NICV distinguisher**

These distinguishers are supported in the distinguisher module of NUSCAR.

<img src="./images/CPA2.png" width=800px>

In [1]:
import nuscar
import numpy as np

In [2]:
reader = nuscar.ReaderZARR("../datasets/aes_stm32.zarr")
ctn = nuscar.ContainerZARR(reader)

### View trace file information

In [3]:
ctn

Trace Count,Sample Points,Meta Info
100,40000,"['plaintext', 'ciphertext']"


In [ ]:
ctn.viewmeta()#View trace contents

In [5]:
ctn.samples[0]

array([-16962, -28553, -32512, ...,  -9611,  -7632,  -5371],
      shape=(40000,), dtype=int16)

In [7]:
ctn.view([0])

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…

### Leakage analysis using a distinguisher

The attacker needs to define a `selection function` that uses the algorithm's `meta` information to compute the intermediate value of interest or its leakage model.

Below, with the key known, we use the Hamming weight of the first-round AES S-box output as the intermediate value for leakage analysis.

In [8]:
key = np.frombuffer(bytes.fromhex("5e75e32656ba6b8c6e26d254f2fc1b44"), dtype='uint8')

In [ ]:
def sf_sbox_out(plaintext, key=key):
    sbox_in = plaintext ^ key # plaintext matches the meta name stored in the trace file; NUSCAR will automatically fetch this meta information and pass it into the function. To speed up computation, the input meta contains multiple traces, i.e. a 2D ndarray array
    sbox_out = nuscar.ciphers.aes.sub_bytes(sbox_in)
    return nuscar.leakmodel.leakage_model_hw(sbox_out, nb_words=1) # Output the Hamming weight

In [ ]:
dist = nuscar.distinguisher.DPADistinguisher() # DPA distinguisher

In [ ]:
task = nuscar.task.DistinguisherTask(ctn, sf_sbox_out, distinguisher=dist) # Define the distinguisher task

In [12]:
task.run()

  0%|          | 0/100 [00:00<?, ?it/s]

2026-05-26 22:35:09,830 - nuscar - INFO - check selection func pass
2026-05-26 22:35:09,830 - nuscar - INFO - check memory pass
2026-05-26 22:35:09,834 - nuscar - INFO - task Dist Task: 100 traces, 1 distinguisher, batch_size=1000, nb_samples=40000
2026-05-26 22:35:09,969 - nuscar - INFO - Task completed, time taken: 0.1 seconds


In [ ]:
print(task.result.shape) # The first dimension is the 16 bytes
print(task.result) #The distinguisher results are recorded in task.result

In [ ]:
task.show_result() # Use show_result() to generate the plot

### Key recovery using a distinguisher

When the key is unknown, the distinguisher computation can be performed by guessing the key, and the correct key can be recovered from the distinguisher output.

The attacker likewise needs to define a `selection function` that uses the algorithm's meta information and the guessed key to compute the intermediate value of interest or its leakage model. Unlike leakage analysis, the `selection function` must now return a 3D ndarray (with an added dimension for the key guess).

NUSCAR provides built-in attack selection functions commonly used for algorithms such as AES:
- **attack_first_sbox_bit**: single-bit value of the first-round AES S-box output
- **attack_first_sbox_hw**: Hamming weight of the first-round AES S-box output
- **attack_first_sbox_value**: output value of the first-round AES S-box
- **attack_last_round_xor_bits**: single-bit value of the XOR between the input and output of the last AES round
- **attack_last_round_xor_hw**: Hamming weight of the XOR between the input and output of the last AES round (Hamming distance model)
- **attack_last_sbox_bits**: single-bit value of the last-round AES S-box output
- **attack_last_sbox_hw**: Hamming weight of the last-round AES S-box output

In [ ]:
ctn = nuscar.ContainerZARR(reader, frame=range(5000, 30000)) # Attack only the frame range

In [16]:
sf_attack_sbox = nuscar.ciphers.aes.attack_first_sbox_hw()

In [ ]:
sf_attack_sbox(ctn[0:10].plaintext).shape # First dimension - the multiple input traces, second dimension - key guesses, third dimension - different byte positions

In [18]:
dist = nuscar.distinguisher.CPADistinguisher()

In [19]:
task = nuscar.task.DistinguisherTask(ctn, sf_attack_sbox, distinguisher=dist)

In [20]:
task.run()

  0%|          | 0/100 [00:00<?, ?it/s]

2026-05-26 22:35:31,795 - nuscar - INFO - check selection func pass
2026-05-26 22:35:31,796 - nuscar - INFO - check memory pass
2026-05-26 22:35:31,800 - nuscar - INFO - task Dist Task: 100 traces, 1 distinguisher, batch_size=1000, nb_samples=25000
2026-05-26 22:36:28,727 - nuscar - INFO - Task completed, time taken: 56.9 seconds


In [ ]:
print(task.result.shape) # First dimension - key guesses, second dimension - different byte positions, third dimension - different sample points of the trace
#The distinguisher results are recorded in task.result

### View candidate key rankings

`show_candidate` displays the candidate key ranking for each target byte in a table. When `frame` is passed, the ranking is computed only within the specified sample interval; `format="hex"` displays candidate keys in hexadecimal, making it easier to compare with the actual AES key.

In [22]:
task.show_candidate(
    top=10,
    correct_key=key,
    frame=range(0, 25000),
    format="hex",
)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
rank 1,0x5e,0x75,0xe3,0x26,0x56,0xba,0x6b,0x8c,0x6e,0x26,0xd2,0x54,0xf2,0xfc,0x1b,0x44
rank 2,0x5f,0x76,0xe1,0x25,0x57,0xb8,0x69,0x8e,0x6f,0x27,0xd0,0x56,0xf3,0xfd,0x18,0x46
rank 3,0x5c,0x84,0x63,0xd8,0x18,0x7c,0x6a,0x8d,0x6d,0x24,0xd3,0x82,0x32,0xff,0x27,0x47
rank 4,0xea,0xd5,0xe2,0x27,0x60,0x76,0x1e,0x8f,0x6c,0xb8,0xd1,0x57,0xf0,0xfe,0xfd,0xd0
rank 5,0xa8,0x6d,0x17,0x24,0xfe,0x71,0x8b,0x95,0xff,0xf0,0xda,0x7f,0x90,0xd8,0xa9,0xf2
rank 6,0xad,0x00,0xc3,0x29,0xb0,0xa4,0x0f,0x37,0xa6,0x4b,0x79,0x4a,0x41,0xdf,0x19,0x24
rank 7,0x51,0xf5,0x3c,0x9c,0xe9,0x8a,0x44,0x79,0x86,0x8d,0xc7,0xa1,0xc4,0x83,0x4a,0xb3
rank 8,0xd7,0x74,0x16,0xfb,0x3a,0xbb,0x86,0x43,0x99,0x6b,0x10,0xd8,0x27,0xf0,0x75,0xc3
rank 9,0x05,0xb4,0x21,0x51,0x6d,0xea,0x12,0xf9,0xbf,0xea,0xd9,0xb2,0x21,0x0a,0xa2,0xfc
rank 10,0x8c,0xcd,0xfd,0x22,0x09,0x3a,0xee,0x66,0xbd,0x84,0xb2,0xae,0x68,0xfa,0xc9,0xb6


### View candidate curves for a single target byte

`show_result(..., plot_type="curve")` displays the CPA score curves of all candidate keys for one target byte. Gray curves are the other candidates; the highlighted curves indicate the best candidate and the correct key candidate.

In [23]:
task.show_result(
    correct_key=key,
    plot_type="curve",
    target_idx=0,
    frame=range(0, 25000),
)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': 'lightgre…

### View the attack result and raw traces together

`plot_result_with_traces` places the attack result curve and the raw traces in subplots that share an x-axis. The first subplot shows the best candidate and correct key curves; the second subplot shows one or more raw traces specified by `trace_idx`.

In [24]:
task.plot_result_with_traces(
    correct_key=key,
    target_idx=0,
    trace_idx=[0, 2, 16],
    frame=range(0, 25000),
)

    'data': [{'hoverinfo': 'name+x+y',
              'line': {'color': '#636EFA'…